# 11 — End-to-end monthly inference + optional detectors + fusion rerun

Runs deployment/src/inference/run_end_to_end_inference.py with auto-picked model bundle and standard output structure.


In [6]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first)."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check, cwd=REPO_ROOT)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = REPO_ROOT / c
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

AOI_PATH = pick_first_existing("deployment/aoi/aoi.geojson", "deployment/aoi/oman_query1.geojson")
START = "2025-01-01"
END   = "2025-03-31"

OUT_ROOT = REPO_ROOT / "deployment/outputs/by_plant/osm_way_386838289"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL = (
    newest_path("runs/fusion/**/*.joblib")
    or newest_path("runs/eval/fusion/**/*.joblib")
)
require_exists(MODEL, "fusion model joblib")

print("AOI:", AOI_PATH)
print("OUT_ROOT:", OUT_ROOT)
print("MODEL:", MODEL)


REPO_ROOT: /Users/ameerfiras/REDNET-ML
AOI: /Users/ameerfiras/REDNET-ML/deployment/aoi/aoi.geojson
OUT_ROOT: /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289
MODEL: /Users/ameerfiras/REDNET-ML/runs/fusion/plants/fusion_alllabels_cv5_v2/fusion_model_cv5.joblib


## 11.1 Print help + run


In [13]:
RUNNER = REPO_ROOT / "deployment/run_end_to_end_inference.py"
print("Runner exists:", RUNNER.exists())
if RUNNER.exists():
    sh(f'python "{RUNNER}" --help', check=False)

FILELISTS_DIR = REPO_ROOT / "data/filelists/8d"
TMP_MODIS_ROOT = REPO_ROOT / "data/l3/tmp_infer"

# sanity
require_exists(FILELISTS_DIR, "filelists_dir")
require_exists(TMP_MODIS_ROOT, "tmp_modis_root")
require_exists(MODEL, "fusion model")

# Full run (uncomment):
# sh(
#   f'python "{RUNNER}" '
#   f'--aoi "{AOI_PATH}" --start "{START}" --end "{END}" '
#   f'--out_root "{OUT_ROOT}" --model "{MODEL}" '
#   f'--filelists_dir "{FILELISTS_DIR}" '
#   f'--tmp_modis_root "{TMP_MODIS_ROOT}" '
#   f'--delete_tmp_each "month" '
#   f'--cloud 30 --per_window 1 --size 640 --stride 256'
# )


Runner exists: True

▶ python "/Users/ameerfiras/REDNET-ML/deployment/run_end_to_end_inference.py" --help
usage: End-to-end monthly HAB inference (space-safe, no retrain) + detectors + fusion
       [-h] --aoi AOI --start START --end END --out_root OUT_ROOT --model
       MODEL [--filelists_dir FILELISTS_DIR] [--max_days MAX_DAYS]
       [--modis_products MODIS_PRODUCTS [MODIS_PRODUCTS ...]]
       [--tmp_modis_root TMP_MODIS_ROOT] [--delete_tmp_each {month,year}]
       [--cloud CLOUD] [--per_window PER_WINDOW] [--size SIZE]
       [--stride STRIDE] [--debug_modis] [--skip_detectors]
       [--detectors_script DETECTORS_SCRIPT] [--fusion_script FUSION_SCRIPT]
       [--det_out_dir DET_OUT_DIR] [--frcnn_r50 FRCNN_R50]
       [--frcnn_mb FRCNN_MB] [--ssd_mb SSD_MB]

options:
  -h, --help            show this help message and exit
  --aoi AOI
  --start START
  --end END
  --out_root OUT_ROOT
  --model MODEL
  --filelists_dir FILELISTS_DIR
                        folder containing filelis

PosixPath('/Users/ameerfiras/REDNET-ML/runs/fusion/plants/fusion_alllabels_cv5_v2/fusion_model_cv5.joblib')

## 11.2 Inspect merged outputs


In [14]:

import pandas as pd
merged = OUT_ROOT / "inference_all_months.csv"
print("merged exists:", merged.exists())
if merged.exists():
    df = pd.read_csv(merged)
    display(df.head(5))
    if "hab_prob" in df.columns:
        print(df["hab_prob"].describe())


merged exists: True


,tile,scene_id,datetime,ndwi_mean,ndwi_std,fai_mean,fai_std,rednir_mean,rednir_std,valid_px,...,sst_anom_x_month_cos_anom_rm_missing,sst_anom_x_month_cos_z_rm_missing,hab_prob,month,p_frcnn_r50_med,p_frcnn_mb_med,p_ssd_mb_med,p_frcnn_r50_med_missing,p_frcnn_mb_med_missing,p_ssd_mb_med_missing
0,S2B_MSIL2A_20250124T064059_R120_T40QFM_2025012...,S2B_MSIL2A_20250124T064059_R120_T40QFM_2025012...,2025-01-24 06:40:59.024000+00:00,-0.052359,1.117587e-08,-0.004054,0.0,0.914774,5.960464e-08,16384,...,0.0,0.0,0.513238,2025-01,0.923458,0.180157,0.382465,0.0,0.0,0.0
1,S2B_MSIL2A_20250124T064059_R120_T40QFM_2025012...,S2B_MSIL2A_20250124T064059_R120_T40QFM_2025012...,2025-01-24 06:40:59.024000+00:00,-0.052359,1.117587e-08,-0.004054,0.0,0.914774,5.960464e-08,16384,...,0.0,0.0,0.513238,2025-01,0.903167,0.222934,0.399053,0.0,0.0,0.0
2,S2C_MSIL2A_20250129T064151_R120_T40QFM_2025012...,S2C_MSIL2A_20250129T064151_R120_T40QFM_2025012...,2025-01-29 06:41:51.025000+00:00,-0.061377,0.000000e+00,0.004246,0.0,0.942828,1.192093e-07,16384,...,0.0,0.0,0.253305,2025-01,0.941082,0.200668,0.434261,0.0,0.0,0.0
3,S2C_MSIL2A_20250129T064151_R120_T40QFM_2025012...,S2C_MSIL2A_20250129T064151_R120_T40QFM_2025012...,2025-01-29 06:41:51.025000+00:00,-0.061377,0.000000e+00,0.004246,0.0,0.942828,1.192093e-07,16384,...,0.0,0.0,0.264551,2025-01,0.874167,0.210099,0.391285,0.0,0.0,0.0
4,S2C_MSIL2A_20250208T064101_R120_T40QFM_2025020...,S2C_MSIL2A_20250208T064101_R120_T40QFM_2025020...,2025-02-08 06:41:01.025000+00:00,-0.099062,1.490116e-08,0.005793,0.0,0.905908,1.192093e-07,16384,...,0.0,0.0,0.422627,2025-02,0.933179,0.204090,0.427748,0.0,0.0,0.0


count    82.000000
mean      0.349393
std       0.125399
min       0.180704
25%       0.250864
50%       0.339722
75%       0.418608
max       0.712871
Name: hab_prob, dtype: float64
